# Toy Word Embedding Training from Scratch

## Learning objective

Train a tiny neural network that learns one static embedding vector for each word by solving a **missing-word prediction** task.

```text
Surrounding words
-> Embedding lookup
-> Average context embeddings
-> Predict the missing middle word
-> Calculate loss
-> Backpropagate
-> Update embeddings
```

This notebook uses only **NumPy** and a small synthetic dataset. The goal is conceptual understanding, not retrieval accuracy.

No API key, internet connection, GPU, LangChain, or Gemini access is required.


## 1. Import the required modules

- `re` tokenizes the simple English text.
- `numpy` stores matrices and performs neural-network calculations.


In [7]:
# !pip install numpy

In [8]:
import re

import numpy as np


np.set_printoptions(
    precision=3,
    suppress=True,
)


## 2. Create a small synthetic corpus

The corpus deliberately repeats related patterns:

```text
customers buy products
customers purchase products
```

Words appearing in similar contexts receive similar training signals.

The repetition increases the number of training steps but does not add new linguistic information.


In [9]:
sentences = [
    "customers buy products",
    "customers purchase products",
    "people buy goods",
    "people purchase goods",
    "shoppers buy products",
    "shoppers purchase goods",
    "companies sell products",
    "stores sell goods",
    "customers visit stores",
    "shoppers visit stores",
    "dogs chase cats",
    "puppies chase cats",
    "dogs like food",
    "puppies like food",
]


sentences = sentences * 100


print("Number of sentence occurrences:", len(sentences))
print("First five sentences:")

for sentence in sentences[:5]:
    print("  ", sentence)


Number of sentence occurrences: 1400
First five sentences:
   customers buy products
   customers purchase products
   people buy goods
   people purchase goods
   shoppers buy products


## 3. Tokenize the sentences

The tokenizer:

1. Converts text to lowercase.
2. Extracts alphabetic word tokens.
3. Returns a list of words.


In [10]:
def tokenize(text):
    """Convert text to lowercase word tokens."""

    return re.findall(
        pattern=r"[a-z]+",
        string=text.lower(),
    )


tokenized_sentences = [
    tokenize(sentence)
    for sentence in sentences
]


for sentence in tokenized_sentences[:5]:
    print(sentence)


['customers', 'buy', 'products']
['customers', 'purchase', 'products']
['people', 'buy', 'goods']
['people', 'purchase', 'goods']
['shoppers', 'buy', 'products']


## 4. Create the vocabulary

A vocabulary contains every distinct word.

Each word receives an integer ID. The IDs are only row numbers and contain no semantic meaning.


In [11]:
all_words = [
    word
    for sentence in tokenized_sentences
    for word in sentence
]


vocabulary = sorted(set(all_words))
vocabulary_size = len(vocabulary)


word_to_id = {
    word: index
    for index, word in enumerate(vocabulary)
}


id_to_word = {
    index: word
    for word, index in word_to_id.items()
}


print("Vocabulary size:", vocabulary_size)
print("Vocabulary:", vocabulary)
print("Word-to-ID mapping:")

for word, word_id in word_to_id.items():
    print(f"  {word:10s} -> {word_id}")


Vocabulary size: 17
Vocabulary: ['buy', 'cats', 'chase', 'companies', 'customers', 'dogs', 'food', 'goods', 'like', 'people', 'products', 'puppies', 'purchase', 'sell', 'shoppers', 'stores', 'visit']
Word-to-ID mapping:
  buy        -> 0
  cats       -> 1
  chase      -> 2
  companies  -> 3
  customers  -> 4
  dogs       -> 5
  food       -> 6
  goods      -> 7
  like       -> 8
  people     -> 9
  products   -> 10
  puppies    -> 11
  purchase   -> 12
  sell       -> 13
  shoppers   -> 14
  stores     -> 15
  visit      -> 16


## 5. Generate missing-word training examples

Every sentence contains three words:

```text
customers buy products
```

The model receives the first and third words as context:

```text
Context: customers, products
```

The target is the missing middle word:

```text
Target: buy
```

This resembles a simplified CBOW or masked-word prediction task.


In [12]:
context_examples = []
target_examples = []


for sentence in tokenized_sentences:

    if len(sentence) != 3:
        continue

    left_word = sentence[0]
    target_word = sentence[1]
    right_word = sentence[2]

    context_ids = [
        word_to_id[left_word],
        word_to_id[right_word],
    ]

    target_id = word_to_id[target_word]

    context_examples.append(context_ids)
    target_examples.append(target_id)


context_examples = np.array(
    context_examples,
    dtype=np.int64,
)


target_examples = np.array(
    target_examples,
    dtype=np.int64,
)


print("Context array shape:", context_examples.shape)
print("Target array shape:", target_examples.shape)


print("First eight training examples:")

for context_ids, target_id in zip(
    context_examples[:8],
    target_examples[:8],
):
    context_words = [
        id_to_word[word_id]
        for word_id in context_ids
    ]

    target_word = id_to_word[target_id]

    print(
        "Context:", context_words,
        "Target:", target_word,
    )


Context array shape: (1400, 2)
Target array shape: (1400,)
First eight training examples:
Context: ['customers', 'products'] Target: buy
Context: ['customers', 'products'] Target: purchase
Context: ['people', 'goods'] Target: buy
Context: ['people', 'goods'] Target: purchase
Context: ['shoppers', 'products'] Target: buy
Context: ['shoppers', 'goods'] Target: purchase
Context: ['companies', 'products'] Target: sell
Context: ['stores', 'goods'] Target: sell


In [13]:
context_examples

array([[ 4, 10],
       [ 4, 10],
       [ 9,  7],
       ...,
       [11,  1],
       [ 5,  6],
       [11,  6]], shape=(1400, 2))

In [14]:
target_examples

array([ 0, 12,  0, ...,  2,  8,  8], shape=(1400,))

## 6. Initialize the trainable parameters

We use five numbers per word.

```text
Embedding matrix shape
= vocabulary size x embedding dimension
```

Every row begins with small random values. After training, each row becomes the learned embedding for one word.

The output layer converts one context vector into one prediction score for every vocabulary word.


In [16]:
random_generator = np.random.default_rng(seed=42)


embedding_dimension = 5


embedding_matrix = random_generator.normal(
    loc=0.0,
    scale=0.1,
    size=(
        vocabulary_size,
        embedding_dimension,
    ),
)


output_weights = random_generator.normal(
    loc=0.0,
    scale=0.1,
    size=(
        embedding_dimension,
        vocabulary_size,
    ),
)


output_bias = np.zeros(vocabulary_size)


print("Embedding matrix shape:", embedding_matrix.shape)
print("Output weight shape:", output_weights.shape)
print("Output bias shape:", output_bias.shape)


print("Initial embedding for 'customers':")
print(embedding_matrix[word_to_id["customers"]])


Embedding matrix shape: (17, 5)
Output weight shape: (5, 17)
Output bias shape: (17,)
Initial embedding for 'customers':
[-0.018 -0.068  0.122 -0.015 -0.043]


## 7. Define softmax

The output layer produces raw scores called **logits**.

Softmax converts the logits into probabilities that sum to 1.

Subtracting the maximum logit improves numerical stability without changing the resulting probabilities.


In [17]:
def softmax(logits):
    """Convert a batch of logits into probabilities."""

    shifted_logits = logits - np.max(
        logits,
        axis=1,
        keepdims=True,
    )

    exponentials = np.exp(shifted_logits)

    probabilities = exponentials / np.sum(
        exponentials,
        axis=1,
        keepdims=True,
    )

    return probabilities


## 8. Inspect one forward pass before training

For every example:

1. Look up the two context-word embeddings.
2. Average them into one context vector.
3. Multiply by the output weights.
4. Add the output bias.
5. Apply softmax.

At initialization, the prediction is close to random.


In [18]:
example_context_ids = context_examples[:1]


example_context_embeddings = embedding_matrix[
    example_context_ids
]


example_context_vector = example_context_embeddings.mean(
    axis=1
)


example_logits = (
    example_context_vector @ output_weights
    + output_bias
)


example_probabilities = softmax(example_logits)


print("Context word IDs:", example_context_ids)
print("Context words:", [
    id_to_word[word_id]
    for word_id in example_context_ids[0]
])
print("Context embeddings shape:", example_context_embeddings.shape)
print("Context vector:", example_context_vector)
print("Probability sum:", example_probabilities.sum())


highest_id = int(np.argmax(example_probabilities[0]))

print("Initial highest-probability word:", id_to_word[highest_id])
print("Correct target word:", id_to_word[target_examples[0]])


Context word IDs: [[ 4 10]]
Context words: ['customers', 'products']
Context embeddings shape: (1, 2, 5)
Context vector: [[ 0.005 -0.002 -0.012 -0.024 -0.045]]
Probability sum: 0.9999999999999999
Initial highest-probability word: food
Correct target word: buy


## 9. Train the neural network

The loop performs:

```text
Forward pass
-> Cross-entropy loss
-> Backpropagation
-> Gradient-descent update
```

The trainable parameters are:

- `embedding_matrix`
- `output_weights`
- `output_bias`

The word embeddings are not calculated manually. They are updated because better embedding values reduce prediction loss.


In [ ]:
learning_rate = 0.05
number_of_epochs = 500
number_of_examples = len(context_examples)


loss_history = []


for epoch in range(number_of_epochs):

    # --------------------------------------------------------
    # FORWARD PASS
    # --------------------------------------------------------

    context_embeddings = embedding_matrix[
        context_examples
    ]

    context_vectors = context_embeddings.mean(
        axis=1
    )

    logits = (
        context_vectors @ output_weights
        + output_bias
    )

    probabilities = softmax(logits)


    # --------------------------------------------------------
    # CROSS-ENTROPY LOSS
    # --------------------------------------------------------

    correct_word_probabilities = probabilities[
        np.arange(number_of_examples),
        target_examples,
    ]

    loss = -np.mean(
        np.log(
            correct_word_probabilities
            + 1e-12
        )
    )

    loss_history.append(float(loss))


    # --------------------------------------------------------
    # BACKPROPAGATION
    # --------------------------------------------------------

    gradient_logits = probabilities.copy()

    gradient_logits[
        np.arange(number_of_examples),
        target_examples,
    ] -= 1

    gradient_logits /= number_of_examples


    gradient_output_weights = (
        context_vectors.T
        @ gradient_logits
    )

    gradient_output_bias = np.sum(
        gradient_logits,
        axis=0,
    )


    gradient_context_vectors = (
        gradient_logits
        @ output_weights.T
    )


    # Two context words were averaged, so each receives
    # one half of the context-vector gradient.

    gradient_each_context_word = (
        gradient_context_vectors / 2
    )


    gradient_embedding_matrix = np.zeros_like(
        embedding_matrix
    )


    for example_index in range(number_of_examples):

        for position_index in range(2):

            word_id = context_examples[
                example_index,
                position_index,
            ]

            gradient_embedding_matrix[
                word_id
            ] += gradient_each_context_word[
                example_index
            ]


    # --------------------------------------------------------
    # PARAMETER UPDATE
    # --------------------------------------------------------

    embedding_matrix -= (
        learning_rate
        * gradient_embedding_matrix
    )

    output_weights -= (
        learning_rate
        * gradient_output_weights
    )

    output_bias -= (
        learning_rate
        * gradient_output_bias
    )


    if epoch % 50 == 0:
        print(
            f"Epoch {epoch:3d}"
            f" | Loss: {loss:.4f}"
        )


print(
    f"Final loss after {number_of_epochs} epochs:"
    f" {loss_history[-1]:.4f}"
)


## 10. Inspect the loss history

The loss should generally decrease. A lower loss means the model assigns more probability to the correct missing words.


In [ ]:
print("Initial loss:", round(loss_history[0], 4))
print("Final loss:", round(loss_history[-1], 4))


print("
Selected loss values:")

for epoch in [0, 50, 100, 200, 300, 400, 499]:
    print(
        f"Epoch {epoch:3d}:"
        f" {loss_history[epoch]:.4f}"
    )


## 11. Retrieve learned embeddings

The embedding for a word is simply the row selected by the word's ID.


In [ ]:
def get_embedding(word):
    """Return the trained embedding vector for one word."""

    word_id = word_to_id[word]
    return embedding_matrix[word_id]


for word in [
    "customers",
    "people",
    "shoppers",
    "products",
    "goods",
    "dogs",
    "puppies",
]:
    print(
        f"{word:10s}",
        get_embedding(word),
    )


## 12. Calculate cosine similarity

Cosine similarity compares vector direction:

```text
1.0  means very similar direction
0.0  means approximately unrelated direction
-1.0 means opposite direction
```

The exact values in this toy example are not important. Look for relative patterns.


In [ ]:
def cosine_similarity(first_vector, second_vector):
    """Calculate cosine similarity between two vectors."""

    numerator = np.dot(
        first_vector,
        second_vector,
    )

    denominator = (
        np.linalg.norm(first_vector)
        * np.linalg.norm(second_vector)
    )

    return numerator / (
        denominator + 1e-12
    )


def compare_words(first_word, second_word):
    """Print similarity between two learned word embeddings."""

    similarity = cosine_similarity(
        get_embedding(first_word),
        get_embedding(second_word),
    )

    print(
        f"{first_word:10s}"
        f" versus "
        f"{second_word:10s}"
        f" = {similarity:.3f}"
    )


compare_words("customers", "shoppers")
compare_words("customers", "people")
compare_words("products", "goods")
compare_words("dogs", "puppies")
compare_words("customers", "dogs")


## 13. Find the nearest words

This compares one query embedding with every other word embedding and sorts by cosine similarity.


In [ ]:
def find_similar_words(
    query_word,
    number_of_results=5,
):
    """Return words nearest to the query embedding."""

    query_embedding = get_embedding(query_word)
    similarities = []

    for candidate_word in vocabulary:

        if candidate_word == query_word:
            continue

        candidate_embedding = get_embedding(
            candidate_word
        )

        similarity = cosine_similarity(
            query_embedding,
            candidate_embedding,
        )

        similarities.append(
            (candidate_word, similarity)
        )

    similarities.sort(
        key=lambda item: item[1],
        reverse=True,
    )

    return similarities[:number_of_results]


for query_word in [
    "customers",
    "products",
    "dogs",
]:
    print(
        f"
Words similar to '{query_word}':"
    )

    for word, similarity in find_similar_words(
        query_word
    ):
        print(
            f"  {word:10s}"
            f" {similarity:.3f}"
        )


## 14. Predict a missing middle word

This repeats the same forward pass used during training, but without backpropagation.

Example:

```text
customers [MASK] products
```


In [ ]:
def predict_missing_word(
    left_word,
    right_word,
    number_of_results=5,
):
    """Predict the missing middle word."""

    context_ids = np.array(
        [
            word_to_id[left_word],
            word_to_id[right_word],
        ]
    )

    context_embeddings = embedding_matrix[
        context_ids
    ]

    context_vector = context_embeddings.mean(
        axis=0
    )

    logits = (
        context_vector @ output_weights
        + output_bias
    )

    probabilities = softmax(
        logits.reshape(1, -1)
    )[0]

    highest_probability_ids = np.argsort(
        probabilities
    )[::-1][:number_of_results]

    predictions = [
        (
            id_to_word[word_id],
            probabilities[word_id],
        )
        for word_id in highest_probability_ids
    ]

    return predictions


predictions = predict_missing_word(
    left_word="customers",
    right_word="products",
)


print("Missing word: customers [MASK] products")

for word, probability in predictions:
    print(
        f"  {word:10s}"
        f" probability = {probability:.3f}"
    )


## 15. Try additional missing-word predictions


In [ ]:
test_contexts = [
    ("people", "goods"),
    ("companies", "products"),
    ("dogs", "cats"),
    ("puppies", "food"),
]


for left_word, right_word in test_contexts:
    print(
        f"
{left_word} [MASK] {right_word}"
    )

    for word, probability in predict_missing_word(
        left_word,
        right_word,
        number_of_results=3,
    ):
        print(
            f"  {word:10s}"
            f" {probability:.3f}"
        )


## 16. What did the model learn?

The model trained three parameter groups:

```text
embedding_matrix
output_weights
output_bias
```

The word embeddings are the rows of `embedding_matrix`.

```python
embedding_matrix[word_to_id["customers"]]
```

The model was never explicitly told that `customers`, `people`, and `shoppers` are related. Similarity can emerge because the words participate in similar sentence patterns.

## Important limitations

- The dataset is tiny and synthetic.
- Repetition does not replace linguistic diversity.
- The model uses a simple average, so word order is lost.
- Every word receives one static vector.
- The model cannot distinguish different meanings of the same word.
- Modern transformer embeddings are contextual and substantially more capable.

## Central takeaway

```text
Random word vectors
-> Predict missing words
-> Measure error
-> Backpropagate
-> Update vectors
-> Similar contexts can produce similar embeddings
```


## 17. Practice exercises

1. Change `embedding_dimension` from 5 to 2 or 10.
2. Add more sentence patterns to the corpus.
3. Remove `sentences = sentences * 100` and compare training.
4. Change the learning rate.
5. Increase or decrease the number of epochs.
6. Add words with two meanings and observe the limitation of one static vector per word.
7. Modify the task to predict surrounding words from a center word, creating a simplified Skip-gram model.
